# Лабораторная работа № 3 — таблицы учебной базы sales

Предварительно выполните предыдущую лабораторную. Создание таблиц рассчитано на первый запуск: при повторном запуске существующие таблицы не удаляются, PostgreSQL сообщит об их наличии. Все изменения сохраняются только в последней ячейке. Если прервали выполнение, выполните `conn.rollback()` и начните заново.

Выполняйте ячейки сверху вниз (**Run All**). Запросы находятся прямо в ноутбуке; SQL-файлы не читаются. Параметры подключения — в следующей ячейке. Зависимости: `pip install "psycopg[binary]" pandas ipykernel` (также поддерживается psycopg2).

[Все лабораторные](../../README.md) · [Отчёты ЛР3](../../labs/lab03/)

## Содержание

- [1. lab3_sales_tables](#part-1)

In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "sales")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: sales


<a id="part-1"></a>

## 1. lab3_sales_tables

Лабораторная работа №3

Создание и связывание таблиц базы данных в среде PostgreSQL

Выполнять после подключения к базе данных sales (\c sales)

Создание таблицы categories

In [2]:
q("""
CREATE TABLE categories (
    cat_ID integer NOT NULL,
    cat_name text NOT NULL,
    cat_overcat integer,
    CONSTRAINT categories_PK PRIMARY KEY (cat_ID)
);
""")

CREATE TABLE


Создание таблицы manufacturers

In [3]:
q("""
CREATE TABLE manufacturers (
    man_ID integer NOT NULL,
    man_name text NOT NULL,
    man_address text NOT NULL,
    CONSTRAINT manufacturers_PK PRIMARY KEY (man_ID)
);
""")

CREATE TABLE


Создание таблицы products

In [4]:
q("""
CREATE TABLE products (
    prod_ID integer NOT NULL,
    prod_name text NOT NULL,
    prod_price numeric(8,2) NOT NULL,
    prod_rest integer NOT NULL,
    prod_cat_ID integer NOT NULL,
    prod_man_ID integer NOT NULL,
    CONSTRAINT products_PK PRIMARY KEY (prod_ID)
);
""")

CREATE TABLE


Создание таблицы customers

In [5]:
q("""
CREATE TABLE customers (
    cust_ID integer NOT NULL,
    cust_name text NOT NULL,
    cust_address text NOT NULL,
    cust_discount integer NOT NULL,
    CONSTRAINT customers_PK PRIMARY KEY (cust_ID)
);
""")

CREATE TABLE


Создание таблицы employees

In [6]:
q("""
CREATE TABLE employees (
    emp_ID integer NOT NULL,
    emp_name text NOT NULL,
    emp_position text NOT NULL,
    CONSTRAINT employees_PK PRIMARY KEY (emp_ID)
);
""")

CREATE TABLE


Создание таблицы orders

In [7]:
q("""
CREATE TABLE orders (
    ord_ID integer NOT NULL,
    ord_cust_ID integer NOT NULL,
    ord_emp_ID integer NOT NULL,
    ord_date date NOT NULL,
    CONSTRAINT orders_PK PRIMARY KEY (ord_ID)
);
""")

CREATE TABLE


Создание таблицы items

In [8]:
q("""
CREATE TABLE items (
    item_ID integer NOT NULL,
    item_ord_ID integer NOT NULL,
    item_prod_ID integer NOT NULL,
    item_prod_count integer NOT NULL,
    CONSTRAINT items_PK PRIMARY KEY (item_ID)
);
""")

CREATE TABLE


Определение внешних ключей

In [9]:
q("""
ALTER TABLE categories
ADD CONSTRAINT categories_FK FOREIGN KEY (cat_overcat)
REFERENCES categories (cat_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [10]:
q("""
ALTER TABLE products
ADD CONSTRAINT products_FK1 FOREIGN KEY (prod_cat_ID)
REFERENCES categories (cat_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [11]:
q("""
ALTER TABLE products
ADD CONSTRAINT products_FK2 FOREIGN KEY (prod_man_ID)
REFERENCES manufacturers (man_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [12]:
q("""
ALTER TABLE orders
ADD CONSTRAINT orders_FK1 FOREIGN KEY (ord_cust_ID)
REFERENCES customers (cust_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [13]:
q("""
ALTER TABLE orders
ADD CONSTRAINT orders_FK2 FOREIGN KEY (ord_emp_ID)
REFERENCES employees (emp_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [14]:
q("""
ALTER TABLE items
ADD CONSTRAINT items_FK1 FOREIGN KEY (item_ord_ID)
REFERENCES orders (ord_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


In [15]:
q("""
ALTER TABLE items
ADD CONSTRAINT items_FK2 FOREIGN KEY (item_prod_ID)
REFERENCES products (prod_ID)
ON DELETE CASCADE
ON UPDATE CASCADE;
""")

ALTER TABLE


## Завершение

Выполните эту ячейку, чтобы завершить транзакцию и закрыть подключение. При ошибке изменения текущей транзакции отменяются; повторите ноутбук с начала.

In [16]:
conn.commit()
conn.close()
print("Работа завершена, изменения сохранены.")

Работа завершена, изменения сохранены.
